# Statistical Filling (Imputation)

<a href="https://colab.research.google.com/github/vuhung16au/ACU-ITEC102/blob/main/Week08/03.Statistical-Filling/notebooks/03-statistical-filling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Overview
When dropping rows deletes too much data or biases results, **imputation** (filling missing values with statistically sound estimates) is the preferred solution.

In this notebook, you will learn how to choose and apply central tendency measures:
- **Mean**: Best for symmetric, normal numerical distributions.
- **Median**: Best when distributions are skewed or contain heavy outliers.
- **Mode**: Best for categorical or discrete attributes.
- **Group-Specific Imputation**: Estimating missing values based on subgroup cohorts.

## 1. Load Dataset & Inspect Null Counts
Let's load `messy_student_data.csv`.

In [ ]:
import pandas as pd

df = pd.read_csv('messy_student_data.csv')
print(f"Loaded {len(df)} student records")
display(df.isna().sum())

## 2. Mean Imputation for Continuous Scores
When numerical data is relatively symmetric, filling with the mean preserves the overall sample average.

In [ ]:
df_mean = df.copy()
avg_score = df_mean['Score'].mean()
print(f"Average Score: {avg_score:.2f}")

df_mean['Score'] = df_mean['Score'].fillna(avg_score)
print(f"Missing scores remaining: {df_mean['Score'].isna().sum()}")

## 3. Median Imputation for Skewed Features
The median (50th percentile) is resilient against extreme outliers and skewed distributions (e.g. Age, Income).

In [ ]:
df_median = df.copy()
median_age = df_median['Age'].median()
print(f"Median Age: {median_age}")

df_median['Age'] = df_median['Age'].fillna(median_age)
print(f"Missing ages remaining: {df_median['Age'].isna().sum()}")

## 4. Mode Imputation for Categoricals
For discrete categories (e.g. Pass/Fail, Campus), fill with the most common value (`.mode()[0]`).

In [ ]:
common_status = df['Status'].mode()[0]
print(f"Most frequent status (mode): {common_status}")

## 5. Group-Specific Imputation with groupby().transform()
Filling missing scores with the mean of each student's specific `Status` cohort (Pass vs Fail) is much more accurate than a single global mean.

In [ ]:
df_group = df.copy()
print("Mean Score by Status:")
display(df_group.groupby('Status')['Score'].mean())

# Impute by group
df_group['Score'] = df_group.groupby('Status')['Score'].transform(
    lambda grp: grp.fillna(grp.mean())
)
print("Remaining null scores:", df_group['Score'].isna().sum())

## 6. Pre- and Post-Imputation Statistical Comparison
Compare descriptive statistics before and after imputation.

In [ ]:
comparison = pd.DataFrame({
    'Raw (with nulls)': df['Score'].describe(),
    'Global Mean Imputed': df_mean['Score'].describe(),
    'Group Mean Imputed': df_group['Score'].describe()
})
display(comparison.round(2))

## Enrichment
### Retaining Missing Flags for Machine Learning
```python
df['Score_Was_Missing'] = df['Score'].isna()
df['Score'] = df['Score'].fillna(df['Score'].mean())
```

## Takeaways
- Imputation retains your full sample size, avoiding the data loss of `.dropna()`.
- Use **mean** for symmetric numerical features without extreme outliers.
- Use **median** for skewed numerical variables (ages, prices, salaries).
- Use **mode** for categorical features.
- Group-specific imputation (`groupby().transform()`) yields more realistic, context-aware values than a blunt global average.

## Conclusion
Statistical filling transforms an incomplete, unworkable dataset into a continuous, usable asset while maintaining aggregate statistical integrity.

## Exercises
**Exercise 1:** Fill missing `Age` values in `df` with the median age, and display the first 5 rows.

**Exercise 2:** Fill missing `Score` values using group-specific mean based on `Status`.

**Exercise 3:** Verify with an assertion that zero missing values remain in both `Age` and `Score`.

In [ ]:
# Write your practice code here

# --- Solutions ---
# clean = df.copy()
# clean['Age'] = clean['Age'].fillna(clean['Age'].median())
# clean['Score'] = clean.groupby('Status')['Score'].transform(lambda g: g.fillna(g.mean()))
# assert clean[['Age', 'Score']].isna().sum().sum() == 0
# print("All missing values successfully imputed!")
# display(clean.head())